# Deepfake Detection — FaceForensics++ Training
**Model:** EfficientNet-B4 (upgraded from B0)  
**Dataset:** FaceForensics++ c40 (Deepfakes vs Original)  
**Goal:** Beat previous Celeb-DF model with trustworthy, reproducible metrics

---
### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your `frames/` folder to Google Drive under `MyDrive/FaceForensics/frames/`
3. Run cells top to bottom

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify frames folder exists
import os
FRAMES_DIR = '/content/drive/MyDrive/FaceForensics/frames'
assert os.path.exists(os.path.join(FRAMES_DIR, 'fake')), 'fake/ folder not found in Drive'
assert os.path.exists(os.path.join(FRAMES_DIR, 'real')), 'real/ folder not found in Drive'

fake_count = len(os.listdir(os.path.join(FRAMES_DIR, 'fake')))
real_count = len(os.listdir(os.path.join(FRAMES_DIR, 'real')))
print(f'Fake frames : {fake_count}')
print(f'Real frames : {real_count}')
print(f'Total       : {fake_count + real_count}')

## Cell 2 — Install & Import

In [ ]:
!pip install -q torchmetrics scikit-learn matplotlib seaborn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import EfficientNet_B4_Weights
from PIL import Image
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3 — Configuration

In [ ]:
CONFIG = {
    # Paths
    'frames_dir'      : '/content/drive/MyDrive/FaceForensics/frames',
    'checkpoint_dir'  : '/content/drive/MyDrive/FaceForensics/checkpoints',
    'output_model'    : '/content/drive/MyDrive/FaceForensics/ff_efficientnet_b4_FINAL.pth',
    'metadata_output' : '/content/drive/MyDrive/FaceForensics/ff_efficientnet_b4_FINAL.json',

    # Model
    'architecture'    : 'EfficientNet-B4',
    'input_size'      : 224,

    # Training
    'epochs'          : 20,
    'batch_size'      : 32,      # T4 can handle B4 at batch 32
    'learning_rate'   : 1e-4,
    'weight_decay'    : 1e-4,
    'patience'        : 5,       # early stopping patience

    # Data split
    'train_ratio'     : 0.70,
    'val_ratio'       : 0.15,
    'test_ratio'      : 0.15,
    'random_seed'     : 42,

    # Inference (keep compatible with existing Flask backend)
    'threshold'       : 0.3,     # same as your existing backend
    'num_frames'      : 10,
}

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
torch.manual_seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])
print('Config loaded.')

## Cell 4 — Dataset with Augmentation

In [ ]:
class FFPlusDataset(Dataset):
    """
    FaceForensics++ frame dataset.
    Label: 0 = real, 1 = fake
    Compatible with existing Flask inference pipeline.
    """
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)


# ImageNet normalization — same as your existing inference.py
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG['input_size'], CONFIG['input_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# No augmentation for val/test — clean evaluation
eval_transform = transforms.Compose([
    transforms.Resize((CONFIG['input_size'], CONFIG['input_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])


def load_dataset(frames_dir, train_ratio, val_ratio, seed):
    """Load all frames, split into train/val/test"""
    all_paths, all_labels = [], []

    for label_name, label_val in [('fake', 1), ('real', 0)]:
        folder = os.path.join(frames_dir, label_name)
        files  = [f for f in os.listdir(folder) if f.endswith('.jpg')]
        for f in files:
            all_paths.append(os.path.join(folder, f))
            all_labels.append(label_val)

    # Shuffle
    rng     = np.random.RandomState(seed)
    indices = rng.permutation(len(all_paths))
    paths   = [all_paths[i]  for i in indices]
    labels  = [all_labels[i] for i in indices]

    n       = len(paths)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    splits = {
        'train': (paths[:n_train],           labels[:n_train]),
        'val'  : (paths[n_train:n_train+n_val], labels[n_train:n_train+n_val]),
        'test' : (paths[n_train+n_val:],     labels[n_train+n_val:])
    }

    print(f'Train : {len(splits["train"][0])} frames')
    print(f'Val   : {len(splits["val"][0])} frames')
    print(f'Test  : {len(splits["test"][0])} frames')

    return splits


splits = load_dataset(
    CONFIG['frames_dir'],
    CONFIG['train_ratio'],
    CONFIG['val_ratio'],
    CONFIG['random_seed']
)

train_dataset = FFPlusDataset(*splits['train'], transform=train_transform)
val_dataset   = FFPlusDataset(*splits['val'],   transform=eval_transform)
test_dataset  = FFPlusDataset(*splits['test'],  transform=eval_transform)

# Weighted sampler to handle any class imbalance
train_labels = splits['train'][1]
class_counts = [train_labels.count(0), train_labels.count(1)]
weights      = [1.0 / class_counts[l] for l in train_labels]
sampler      = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'\nClass balance — Real: {class_counts[0]}, Fake: {class_counts[1]}')

## Cell 5 — Model (Drop-in Compatible with Existing Backend)

In [ ]:
class DeepfakeDetector(nn.Module):
    """
    EfficientNet-B4 deepfake detector.
    Drop-in replacement for existing EfficientNet-B0 model.
    Same output format: single sigmoid value (0=real, 1=fake)
    """
    def __init__(self, pretrained=True):
        super(DeepfakeDetector, self).__init__()

        if pretrained:
            weights = EfficientNet_B4_Weights.DEFAULT
            self.backbone = models.efficientnet_b4(weights=weights)
        else:
            self.backbone = models.efficientnet_b4(weights=None)

        # Same classifier structure as your existing model
        in_features = self.backbone.classifier[1].in_features  # 1792 for B4 vs 1280 for B0
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4),           # slightly higher dropout for larger model
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.backbone(x)


model = DeepfakeDetector(pretrained=True).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

## Cell 6 — Training Setup

In [ ]:
criterion = nn.BCELoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# Reduce LR when val loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    verbose=True
)

print('Optimizer : AdamW')
print('Scheduler : ReduceLROnPlateau (factor=0.5, patience=2)')
print('Loss      : BCELoss')

## Cell 7 — Training Loop with Early Stopping

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds       = (outputs > CONFIG['threshold']).float()
        correct    += (preds == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_scores = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds       = (outputs > CONFIG['threshold']).float()
            correct    += (preds == labels).sum().item()
            total      += images.size(0)

            all_scores.extend(outputs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(labels.cpu().numpy().flatten())

    auc = roc_auc_score(all_labels, all_scores)
    return total_loss / total, correct / total, auc, all_preds, all_labels, all_scores


# Training loop
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}

best_val_loss  = float('inf')
patience_count = 0
best_epoch     = 0

print(f'Starting training — {CONFIG["epochs"]} epochs max, early stopping patience={CONFIG["patience"]}')
print('='*70)

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_auc, _, _, _ = eval_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    print(f'Epoch {epoch:02d}/{CONFIG["epochs"]} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} AUC: {val_auc:.4f}')

    # Save checkpoint every epoch to Drive
    ckpt_path = os.path.join(CONFIG['checkpoint_dir'], f'epoch_{epoch:02d}.pth')
    torch.save({
        'epoch'             : epoch,
        'model_state_dict'  : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss'          : val_loss,
        'val_acc'           : val_acc,
        'val_auc'           : val_auc,
    }, ckpt_path)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_epoch     = epoch
        patience_count = 0
        torch.save({
            'epoch'             : epoch,
            'model_state_dict'  : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss'          : val_loss,
            'val_acc'           : val_acc,
            'val_auc'           : val_auc,
        }, CONFIG['output_model'])
        print(f'  ✓ Best model saved (val_loss={val_loss:.4f})')
    else:
        patience_count += 1
        print(f'  No improvement. Patience: {patience_count}/{CONFIG["patience"]}')
        if patience_count >= CONFIG['patience']:
            print(f'\nEarly stopping at epoch {epoch}. Best epoch was {best_epoch}.')
            break

print('\nTraining complete.')

## Cell 8 — Training Curves

In [ ]:
epochs_ran = len(history['train_loss'])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Curves — EfficientNet-B4 on FF++', fontsize=14)

axes[0].plot(x, history['train_loss'], label='Train', marker='o')
axes[0].plot(x, history['val_loss'],   label='Val',   marker='o')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(x, history['train_acc'], label='Train', marker='o')
axes[1].plot(x, history['val_acc'],   label='Val',   marker='o')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(x, history['val_auc'], label='Val AUC', marker='o', color='green')
axes[2].set_title('AUC-ROC')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
curves_path = '/content/drive/MyDrive/FaceForensics/training_curves.png'
plt.savefig(curves_path, dpi=150)
plt.show()
print(f'Saved: {curves_path}')

## Cell 9 — Final Evaluation on Test Set

In [ ]:
# Load best model for test evaluation
checkpoint = torch.load(CONFIG['output_model'], map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]}')

_, test_acc, test_auc, test_preds, test_labels, test_scores = eval_epoch(
    model, test_loader, criterion, device
)

precision = precision_score(test_labels, test_preds)
recall    = recall_score(test_labels, test_preds)
f1        = f1_score(test_labels, test_preds)
cm        = confusion_matrix(test_labels, test_preds)

print('\n' + '='*50)
print('TEST SET RESULTS')
print('='*50)
print(f'Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'AUC-ROC   : {test_auc:.4f}')
print(f'\nConfusion Matrix:')
print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
print(f'  FN={cm[1,0]}  TP={cm[1,1]}')

## Cell 10 — Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Test Set Evaluation — EfficientNet-B4 on FF++', fontsize=14)

# Confusion Matrix
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Real', 'Fake'],
    yticklabels=['Real', 'Fake']
)
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(test_labels, test_scores)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {test_auc:.4f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
eval_path = '/content/drive/MyDrive/FaceForensics/evaluation_plots.png'
plt.savefig(eval_path, dpi=150)
plt.show()
print(f'Saved: {eval_path}')

## Cell 11 — Save Metadata JSON (for Flask /model-info endpoint)

In [ ]:
# This JSON is loaded by your existing /model-info Flask endpoint
metadata = {
    'model_name'    : 'Deepfake Detector v3.0',
    'architecture'  : 'EfficientNet-B4',
    'dataset'       : 'FaceForensics++ c40 (Deepfakes vs Original)',
    'trained_at'    : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'best_epoch'    : checkpoint['epoch'],

    'performance_metrics': {
        'test_accuracy' : f'{test_acc*100:.2f}%',
        'auc_score'     : f'{test_auc*100:.2f}%',
        'precision'     : f'{precision*100:.2f}%',
        'recall'        : f'{recall*100:.2f}%',
        'f1_score'      : f'{f1*100:.2f}%',
    },

    'training_details': {
        'dataset'          : 'FaceForensics++ c40',
        'total_frames'     : len(train_dataset) + len(val_dataset) + len(test_dataset),
        'train_frames'     : len(train_dataset),
        'val_frames'       : len(val_dataset),
        'test_frames'      : len(test_dataset),
        'epochs_trained'   : epochs_ran,
        'batch_size'       : CONFIG['batch_size'],
        'optimizer'        : 'AdamW',
        'augmentation'     : 'HorizontalFlip, ColorJitter, Rotation',
    },

    'inference_settings': {
        'default_frames_analyzed': CONFIG['num_frames'],
        'frame_sampling'         : 'evenly distributed',
        'input_size'             : '224x224',
        'prediction_threshold'   : CONFIG['threshold'],
    }
}

with open(CONFIG['metadata_output'], 'w') as f:
    json.dump(metadata, f, indent=2)

print('Metadata saved to Drive.')
print(json.dumps(metadata['performance_metrics'], indent=2))

## Cell 12 — How to Use in Your Existing Flask Backend

Only **3 changes** needed in your `inference.py`:

```python
# 1. Import B4 instead of B0
from torchvision.models import EfficientNet_B4_Weights

# 2. In DeepfakeDetector.__init__(), change:
self.backbone = models.efficientnet_b4(weights=weights)   # was efficientnet_b0

# 3. In app.py, update model path:
MODEL_PATH = 'data/models/ff_efficientnet_b4_FINAL.pth'
```

Everything else — `predict_video()`, Flask endpoints, React frontend — stays exactly the same.

---

### Files to download from Drive after training:
- `ff_efficientnet_b4_FINAL.pth` → replace your existing `.pth` file
- `ff_efficientnet_b4_FINAL.json` → place next to `.pth` for `/model-info` endpoint
- `training_curves.png` → for GitHub README
- `evaluation_plots.png` → for GitHub README